[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CityScope/TransitLOSStudies/blob/main/example_metro_area/example.ipynb)

# Transit level of service for any area

1. Choose an area: type a place name and an area type (UN functional area by default), draw it on a map, or upload a file.
2. Choose the day, the hours and the transit feeds.
3. Run the study.
4. See the results on an interactive map and download a zip with the streets, the H3 hexagons and the census.

Every input comes from the most local source available: the area's own census and official transit feeds when there are, otherwise global data (WorldPop, Mobility Database, Transitland).

The level of service uses the standard scoring functions of the transitLOS package, the ones taken from the transit literature (stop score from the walking decay, headway, speed and mode). No curve is fitted to data here.

Run the cells in order. In Google Colab: **Runtime → Run all** also works with the defaults (Boston).

## 1. Install

In [ ]:
# Gets the study code, installs what is missing (Colab: the session restarts once, then run all cells again).
import subprocess, sys
from pathlib import Path
if not Path("bootstrap.py").is_file():
    subprocess.check_call(["git", "clone", "-q", "--depth", "1", "https://github.com/CityScope/TransitLOSStudies.git"])
    sys.path.insert(0, "TransitLOSStudies/example_metro_area")
import bootstrap
bootstrap.setup()
import example

## 2. Choose the area

Use **one** of the three options below. The last one you run is used.

### Option A: a place name and an area type
`AREA` can be:
- `"metro_area"` (also `"functional_area"`): the UN functional urban area, the urban centre plus its commuting zone (default)
- `"urban_core"`: the UN urban centre (the dense built-up city)
- `"municipality"`: the city's own boundary (the official one when the country publishes it)

In [ ]:
PLACE = "Boston, Massachusetts, USA"
AREA = "metro_area"

aoi = example.area_from_place(PLACE, AREA)
print(aoi["name"].iloc[0], f"{example.area_km2(aoi):,.0f} km²")

### Option B: draw it on a map
Draw a polygon or rectangle with the tools on the left of the map, then run the next cell.

In [ ]:
draw_map, drawn = example.draw_area(aoi)
draw_map

In [ ]:
if drawn:
    aoi = example.area_from_geojson(drawn[-1])
else:
    print("Nothing drawn: keeping the area from option A")

### Option C: upload a file
Any file with a boundary: `.gpkg`, `.geojson`, `.kml`, or a zipped shapefile. Set `UPLOAD = True` to pick it from your computer.

In [ ]:
UPLOAD = False        # True: choose a file from your computer (Colab)
AOI_FILE = None       # or the path of a file, e.g. "my_area.gpkg"
aoi = example.area_from_upload(UPLOAD, AOI_FILE, aoi)

## 3. Check and confirm the area

The map shows the area that will be analysed. Tick the box to confirm; the study does not start until you do. To change the area, go back to section 2 and run this cell again.

In [ ]:
confirm = example.confirm_area(aoi)

## 4. Choose the day and the hours

In [ ]:
DAY = "weekday"         # "weekday" (a working day), "saturday" or "sunday"
START_TIME = "06:00"
END_TIME = "22:00"

## 5. Choose the transit feeds

- `"download"`: find the area's feeds online: the country's official portal when there is one, plus the feeds it misses from the Mobility Database and Transitland.
  If a source needs an API key, the run asks for it and says where to get one. Press Enter to skip that source.
- A folder with your GTFS feeds, e.g. `"my_feeds/"`.
- A list of feeds: zip files, folders or download links.

In [ ]:
FEEDS = "download"
# FEEDS = "my_feeds/"
# FEEDS = ["bus_gtfs.zip", "https://example.com/rail_gtfs.zip"]
# FEEDS = example.upload_feeds()      # Colab: choose GTFS zip files from your computer

## 6. Run

This gets the streets, the transit feeds and the census for the area, then computes everything.
A municipality takes from a few minutes to an hour; a whole metro area takes longer.

In [ ]:
zip_path = example.run_confirmed(confirm, aoi, day=DAY, start_time=START_TIME, end_time=END_TIME, feeds=FEEDS)

## 7. See the results on the map

The map shows the finest H3 hexagons or circles and the finest census level, with the stops and routes. The chart button (bottom left) opens the distribution, regression, ANOVA, discretization and metadata tabs.

In [ ]:
example.show_map(zip_path)

## 8. Download the results

The zip holds:
- `streets.parquet`: every street segment with its transit level of service
- `h3_res<N>.parquet`: H3 hexagons at each resolution, with population, level of service and census columns
- `census_<level>.parquet`: the census at its finest level, with the same columns

In [ ]:
example.download(zip_path)